In [ ]:
# 1. Strict Environment Setup & GPU Enforcement
import os, jax, logging, pymc.sampling.jax

os.environ.update({
    "OMP_NUM_THREADS": "1", 
    "OPENBLAS_NUM_THREADS": "1", 
    "MKL_NUM_THREADS": "1", 
    "VECLIB_MAXIMUM_THREADS": "1", 
    "NUMEXPR_NUM_THREADS": "1", 
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false", 
    "XLA_PYTHON_CLIENT_ALLOCATOR": "platform"
})
logging.getLogger("pymc").setLevel(logging.ERROR)
jax.config.update("jax_platform_name", "cuda")

import pandas as pd
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import jax.numpy as jnp
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set Working Directory (WSL Rclone path)
work_dir = "/home/daves/google_drive/Pessoal/Notebooks/2026 World Cup"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# 2. Data Downloading & Preprocessing
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
df = pd.read_csv(url)
df['date'] = pd.to_datetime(df['date'])

# Inject the 2026 official scheduled matches into the dataset if they aren't there yet.
# We map out the games up to June 16 to check against the real-time date.
scheduled_matches_2026 = pd.DataFrame([
    # ---- MATCHDAY 1 (June 11-17) ----
    # Group A
    {'date': '2026-06-11', 'home_team': 'Mexico',               'away_team': 'South Africa',          'tournament': 'FIFA World Cup', 'neutral': False},
    {'date': '2026-06-11', 'home_team': 'South Korea',           'away_team': 'Czech Republic',        'tournament': 'FIFA World Cup', 'neutral': True},
    # Group B
    {'date': '2026-06-12', 'home_team': 'Canada',                'away_team': 'Bosnia and Herzegovina','tournament': 'FIFA World Cup', 'neutral': False},
    # Group D
    {'date': '2026-06-12', 'home_team': 'United States',         'away_team': 'Paraguay',              'tournament': 'FIFA World Cup', 'neutral': False},
    {'date': '2026-06-13', 'home_team': 'Australia',             'away_team': 'Turkey',                'tournament': 'FIFA World Cup', 'neutral': True},
    # Group C
    {'date': '2026-06-13', 'home_team': 'Haiti',                 'away_team': 'Scotland',              'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-13', 'home_team': 'Brazil',                'away_team': 'Morocco',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group B
    {'date': '2026-06-13', 'home_team': 'Qatar',                 'away_team': 'Switzerland',           'tournament': 'FIFA World Cup', 'neutral': True},
    # Group E
    {'date': '2026-06-14', 'home_team': 'Ivory Coast',           'away_team': 'Ecuador',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-14', 'home_team': 'Germany',               'away_team': 'Curaçao',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group F
    {'date': '2026-06-14', 'home_team': 'Netherlands',           'away_team': 'Japan',                 'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-14', 'home_team': 'Sweden',                'away_team': 'Tunisia',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group H
    {'date': '2026-06-15', 'home_team': 'Saudi Arabia',          'away_team': 'Uruguay',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-15', 'home_team': 'Spain',                 'away_team': 'Cape Verde',            'tournament': 'FIFA World Cup', 'neutral': True},
    # Group G
    {'date': '2026-06-15', 'home_team': 'Iran',                  'away_team': 'New Zealand',           'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-15', 'home_team': 'Belgium',               'away_team': 'Egypt',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group I
    {'date': '2026-06-16', 'home_team': 'France',                'away_team': 'Senegal',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-16', 'home_team': 'Iraq',                  'away_team': 'Norway',                'tournament': 'FIFA World Cup', 'neutral': True},
    # Group J
    {'date': '2026-06-16', 'home_team': 'Argentina',             'away_team': 'Algeria',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-16', 'home_team': 'Austria',               'away_team': 'Jordan',                'tournament': 'FIFA World Cup', 'neutral': True},
    # Group K
    {'date': '2026-06-17', 'home_team': 'Portugal',              'away_team': 'DR Congo',              'tournament': 'FIFA World Cup', 'neutral': True},
    # Group L
    {'date': '2026-06-17', 'home_team': 'England',               'away_team': 'Croatia',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-17', 'home_team': 'Ghana',                 'away_team': 'Panama',                'tournament': 'FIFA World Cup', 'neutral': True},
    # Group K
    {'date': '2026-06-17', 'home_team': 'Uzbekistan',            'away_team': 'Colombia',              'tournament': 'FIFA World Cup', 'neutral': True},
    # ---- MATCHDAY 2 (June 18-23) ----
    # Group A
    {'date': '2026-06-18', 'home_team': 'Czech Republic',        'away_team': 'South Africa',          'tournament': 'FIFA World Cup', 'neutral': True},
    # Group B
    {'date': '2026-06-18', 'home_team': 'Switzerland',           'away_team': 'Bosnia and Herzegovina','tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-18', 'home_team': 'Canada',                'away_team': 'Qatar',                 'tournament': 'FIFA World Cup', 'neutral': False},
    # Group A
    {'date': '2026-06-18', 'home_team': 'Mexico',                'away_team': 'South Korea',           'tournament': 'FIFA World Cup', 'neutral': False},
    # Group D
    {'date': '2026-06-19', 'home_team': 'United States',         'away_team': 'Australia',             'tournament': 'FIFA World Cup', 'neutral': False},
    # Group C
    {'date': '2026-06-19', 'home_team': 'Scotland',              'away_team': 'Morocco',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-19', 'home_team': 'Brazil',                'away_team': 'Haiti',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group D
    {'date': '2026-06-19', 'home_team': 'Turkey',                'away_team': 'Paraguay',              'tournament': 'FIFA World Cup', 'neutral': True},
    # Group F
    {'date': '2026-06-20', 'home_team': 'Netherlands',           'away_team': 'Sweden',                'tournament': 'FIFA World Cup', 'neutral': True},
    # Group E
    {'date': '2026-06-20', 'home_team': 'Germany',               'away_team': 'Ivory Coast',           'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-20', 'home_team': 'Ecuador',               'away_team': 'Curaçao',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group F
    {'date': '2026-06-20', 'home_team': 'Tunisia',               'away_team': 'Japan',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group H
    {'date': '2026-06-21', 'home_team': 'Spain',                 'away_team': 'Saudi Arabia',          'tournament': 'FIFA World Cup', 'neutral': True},
    # Group G
    {'date': '2026-06-21', 'home_team': 'Belgium',               'away_team': 'Iran',                  'tournament': 'FIFA World Cup', 'neutral': True},
    # Group H
    {'date': '2026-06-21', 'home_team': 'Uruguay',               'away_team': 'Cape Verde',            'tournament': 'FIFA World Cup', 'neutral': True},
    # Group G
    {'date': '2026-06-21', 'home_team': 'New Zealand',           'away_team': 'Egypt',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group J
    {'date': '2026-06-22', 'home_team': 'Argentina',             'away_team': 'Austria',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group I
    {'date': '2026-06-22', 'home_team': 'France',                'away_team': 'Iraq',                  'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-22', 'home_team': 'Norway',                'away_team': 'Senegal',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group J
    {'date': '2026-06-22', 'home_team': 'Jordan',                'away_team': 'Algeria',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group K
    {'date': '2026-06-23', 'home_team': 'Portugal',              'away_team': 'Uzbekistan',            'tournament': 'FIFA World Cup', 'neutral': True},
    # Group L
    {'date': '2026-06-23', 'home_team': 'England',               'away_team': 'Ghana',                 'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-23', 'home_team': 'Panama',                'away_team': 'Croatia',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group K
    {'date': '2026-06-23', 'home_team': 'Colombia',              'away_team': 'DR Congo',              'tournament': 'FIFA World Cup', 'neutral': True},
    # ---- MATCHDAY 3 (June 24-27) — simultaneous pairs ----
    # Group B
    {'date': '2026-06-24', 'home_team': 'Switzerland',           'away_team': 'Canada',                'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-24', 'home_team': 'Bosnia and Herzegovina','away_team': 'Qatar',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group A
    {'date': '2026-06-24', 'home_team': 'Czech Republic',        'away_team': 'Mexico',                'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-24', 'home_team': 'South Africa',          'away_team': 'South Korea',           'tournament': 'FIFA World Cup', 'neutral': True},
    # Group C
    {'date': '2026-06-24', 'home_team': 'Scotland',              'away_team': 'Brazil',                'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-24', 'home_team': 'Morocco',               'away_team': 'Haiti',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group E
    {'date': '2026-06-25', 'home_team': 'Ecuador',               'away_team': 'Germany',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-25', 'home_team': 'Curaçao',               'away_team': 'Ivory Coast',           'tournament': 'FIFA World Cup', 'neutral': True},
    # Group F
    {'date': '2026-06-25', 'home_team': 'Japan',                 'away_team': 'Sweden',                'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-25', 'home_team': 'Tunisia',               'away_team': 'Netherlands',           'tournament': 'FIFA World Cup', 'neutral': True},
    # Group D
    {'date': '2026-06-25', 'home_team': 'Turkey',                'away_team': 'United States',         'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-25', 'home_team': 'Paraguay',              'away_team': 'Australia',             'tournament': 'FIFA World Cup', 'neutral': True},
    # Group I
    {'date': '2026-06-26', 'home_team': 'Norway',                'away_team': 'France',                'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-26', 'home_team': 'Senegal',               'away_team': 'Iraq',                  'tournament': 'FIFA World Cup', 'neutral': True},
    # Group H
    {'date': '2026-06-26', 'home_team': 'Cape Verde',            'away_team': 'Saudi Arabia',          'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-26', 'home_team': 'Uruguay',               'away_team': 'Spain',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group G
    {'date': '2026-06-26', 'home_team': 'Egypt',                 'away_team': 'Iran',                  'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-26', 'home_team': 'New Zealand',           'away_team': 'Belgium',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group L
    {'date': '2026-06-27', 'home_team': 'Panama',                'away_team': 'England',               'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-27', 'home_team': 'Croatia',               'away_team': 'Ghana',                 'tournament': 'FIFA World Cup', 'neutral': True},
    # Group J
    {'date': '2026-06-27', 'home_team': 'Jordan',                'away_team': 'Argentina',             'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-27', 'home_team': 'Algeria',               'away_team': 'Austria',               'tournament': 'FIFA World Cup', 'neutral': True},
    # Group K
    {'date': '2026-06-27', 'home_team': 'DR Congo',              'away_team': 'Uzbekistan',            'tournament': 'FIFA World Cup', 'neutral': True},
    {'date': '2026-06-27', 'home_team': 'Colombia',              'away_team': 'Portugal',              'tournament': 'FIFA World Cup', 'neutral': True},
])
scheduled_matches_2026['date'] = pd.to_datetime(scheduled_matches_2026['date'])

# Merge scheduled matches into df if they don't exist yet (preventing duplicates)
df = df.merge(scheduled_matches_2026, on=['date', 'home_team', 'away_team', 'tournament', 'neutral'], how='outer', suffixes=('', '_y'))
if 'home_score_y' in df.columns:
    df['home_score'] = df['home_score'].combine_first(df['home_score_y'])
    df['away_score'] = df['away_score'].combine_first(df['away_score_y'])
    df = df.drop(columns=['home_score_y', 'away_score_y'])

# --- Interactive Missing Matches Logic ---
today = pd.Timestamp.today().normalize()

# Filter for matches that should have been played by today but are missing scores
missing_matches = df[(df['date'] <= today) & (df['tournament'] == 'FIFA World Cup') & (df['date'] >= '2026-06-11') & (df['date'] <= '2026-06-27') & (df['home_score'].isna() | df['away_score'].isna())].copy()

if not missing_matches.empty:
    print(f"\n[!] Today is {today.date()}. There are matches played up to today missing results.")
    print("Please enter the final scores for the following matches.")
    print("Leave blank and press Enter to skip a specific match.\n")
    
    for index, row in missing_matches.iterrows():
        t1 = row['home_team']
        t2 = row['away_team']
        match_date = row['date'].date()
        
        while True:
            user_input = input(f"[{match_date}] {t1} vs {t2} (Format: Score A, Score B): ")
            if not user_input.strip():
                print(f"Skipping {t1} vs {t2}...")
                break
            
            try:
                parts = [p.strip() for p in user_input.split(',')]
                if len(parts) == 2:
                    s1, s2 = int(parts[0]), int(parts[1])
                    
                    # Update the global dataframe
                    df.loc[index, 'home_score'] = s1
                    df.loc[index, 'away_score'] = s2
                    print(f"--> Saved: {t1} {s1} - {s2} {t2}\n")
                    break
                else:
                    print("Invalid format. Please use 'Score A, Score B' (e.g., '2, 1').")
            except Exception as e:
                print(f"Error parsing input. Please use numbers only: {e}")

# Re-isolate completed matches for the 2026 World Cup after updating
wc_2026_completed = df[(df['date'] >= '2026-06-11') & (df['date'] <= '2026-06-27') & (df['tournament'] == 'FIFA World Cup') & df['home_score'].notna()].copy()
wc_2026_completed['home_score'] = wc_2026_completed['home_score'].astype(int)
wc_2026_completed['away_score'] = wc_2026_completed['away_score'].astype(int)

# Filter global dataset strictly for form AFTER the 2022 World Cup Final (Dec 18, 2022)
df_model = df[(df['date'] > '2022-12-18') & df['home_score'].notna()].copy()
df_model['home_score'] = df_model['home_score'].astype(int)
df_model['away_score'] = df_model['away_score'].astype(int)

teams = list(set(df_model['home_team'].unique()) | set(df_model['away_team'].unique()))
team_to_idx = {team: i for i, team in enumerate(teams)}

df_model['home_idx'] = df_model['home_team'].map(team_to_idx)
df_model['away_idx'] = df_model['away_team'].map(team_to_idx)

# Identify neutral matches to extract raw, uninflated team strengths
is_home_match = (~df_model['neutral']).astype(int).values

# 3. Bayesian Model: Team Strengths Inference (Neutral-Aware)
print("\nRunning Bayesian Inference on GPU...")
with pm.Model() as model:
    atts = pm.Normal('atts', mu=0, sigma=2, shape=len(teams))
    defs = pm.Normal('defs', mu=0, sigma=2, shape=len(teams))
    home_adv = pm.Normal('home_adv', mu=0, sigma=1)
    intercept = pm.Normal('intercept', mu=0, sigma=2)
    
    # Home advantage is only factored in if the match was not played at a neutral venue
    home_theta = pt.exp(intercept + (home_adv * is_home_match) + atts[df_model['home_idx'].values] - defs[df_model['away_idx'].values])
    away_theta = pt.exp(intercept + atts[df_model['away_idx'].values] - defs[df_model['home_idx'].values])
    
    pm.Poisson('home_goals', mu=home_theta, observed=df_model['home_score'].values)
    pm.Poisson('away_goals', mu=away_theta, observed=df_model['away_score'].values)
    
    idata = pymc.sampling.jax.sample_blackjax_nuts(
        draws=5000, 
        tune=5000, 
        chains=4, 
        chain_method="vectorized", 
        progressbar=False, 
        random_seed=42
    )

# 4. Extract Parameters & Define 2026 Schedule
post = idata.posterior
intercept_mean = float(post['intercept'].mean())
home_adv_mean = float(post['home_adv'].mean())
att_mean = post['atts'].mean(dim=['chain', 'draw']).values
def_mean = post['defs'].mean(dim=['chain', 'draw']).values

tournament_teams = [
    'Mexico', 'South Africa', 'South Korea', 'Czech Republic',
    'Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland',
    'Brazil', 'Morocco', 'Haiti', 'Scotland',
    'United States', 'Paraguay', 'Australia', 'Turkey',
    'Germany', 'Curaçao', 'Ivory Coast', 'Ecuador',
    'Netherlands', 'Japan', 'Sweden', 'Tunisia',
    'Belgium', 'Egypt', 'Iran', 'New Zealand',
    'Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay',
    'France', 'Senegal', 'Iraq', 'Norway',
    'Argentina', 'Algeria', 'Austria', 'Jordan',
    'Portugal', 'DR Congo', 'Uzbekistan', 'Colombia',
    'England', 'Croatia', 'Ghana', 'Panama'
]

# Population mapping (in millions) to act as a proxy for "Home Advantage"
population_millions = {
    'Mexico': 128.5, 'South Africa': 60.0, 'South Korea': 51.7, 'Czech Republic': 10.5,
    'Canada': 38.2, 'Bosnia and Herzegovina': 3.2, 'Qatar': 2.7, 'Switzerland': 8.7,
    'Brazil': 214.3, 'Morocco': 37.5, 'Haiti': 11.5, 'Scotland': 5.5,
    'United States': 331.9, 'Paraguay': 6.7, 'Australia': 25.7, 'Turkey': 85.3,
    'Germany': 83.2, 'Curaçao': 0.15, 'Ivory Coast': 27.5, 'Ecuador': 17.8,
    'Netherlands': 17.5, 'Japan': 125.7, 'Sweden': 10.4, 'Tunisia': 12.0,
    'Belgium': 11.6, 'Egypt': 109.3, 'Iran': 87.9, 'New Zealand': 5.1,
    'Spain': 47.4, 'Cape Verde': 0.6, 'Saudi Arabia': 35.3, 'Uruguay': 3.4,
    'France': 67.7, 'Senegal': 16.9, 'Iraq': 43.5, 'Norway': 5.4,
    'Argentina': 45.8, 'Algeria': 44.2, 'Austria': 9.0, 'Jordan': 11.1,
    'Portugal': 10.3, 'DR Congo': 95.9, 'Uzbekistan': 35.1, 'Colombia': 51.5,
    'England': 56.0, 'Croatia': 4.0, 'Ghana': 32.8, 'Panama': 4.3
}

t1_list, t2_list = [], []
for g in range(12):
    for i in range(4):
        for j in range(i+1, 4):
            t1_list.append(tournament_teams[g*4 + i])
            t2_list.append(tournament_teams[g*4 + j])

actual_g1 = np.full(72, -1.0)
actual_g2 = np.full(72, -1.0)

for idx in range(72):
    t1, t2 = t1_list[idx], t2_list[idx]
    match = wc_2026_completed[((wc_2026_completed['home_team'] == t1) & (wc_2026_completed['away_team'] == t2)) | 
                              ((wc_2026_completed['home_team'] == t2) & (wc_2026_completed['away_team'] == t1))]
    if not match.empty:
        row = match.iloc[-1] # Take the most recently added result if duplicates exist
        if row['home_team'] == t1:
            actual_g1[idx], actual_g2[idx] = row['home_score'], row['away_score']
        else:
            actual_g1[idx], actual_g2[idx] = row['away_score'], row['home_score']

actual_g1_jax = jnp.array(actual_g1)
actual_g2_jax = jnp.array(actual_g2)

tourney_indices = np.array([team_to_idx.get(team, 0) for team in tournament_teams])
tourney_att = jnp.array(att_mean[tourney_indices])
tourney_def = jnp.array(def_mean[tourney_indices])

t1_idx = jnp.array([tournament_teams.index(t) for t in t1_list])
t2_idx = jnp.array([tournament_teams.index(t) for t in t2_list])

t1_pops = jnp.array([population_millions.get(t, 0) for t in t1_list])
t2_pops = jnp.array([population_millions.get(t, 0) for t in t2_list])

is_t1_larger = (t1_pops > t2_pops)
is_t2_larger = (t2_pops > t1_pops)

# 5. Fast Monte Carlo Simulation on GPU (Skipping played matches logic via masking)
N_sims = 100000

log_lam1 = intercept_mean + tourney_att[t1_idx] - tourney_def[t2_idx] + jnp.where(is_t1_larger, home_adv_mean, 0.0)
log_lam2 = intercept_mean + tourney_att[t2_idx] - tourney_def[t1_idx] + jnp.where(is_t2_larger, home_adv_mean, 0.0)

key = jax.random.PRNGKey(42)
key1, key2 = jax.random.split(key)

sim_goals1 = jax.random.poisson(key1, jnp.exp(log_lam1), shape=(N_sims, 72))
sim_goals2 = jax.random.poisson(key2, jnp.exp(log_lam2), shape=(N_sims, 72))

# JAX statically masks out the random numbers where a match has already been played
goals1 = jnp.where(actual_g1_jax != -1, actual_g1_jax, sim_goals1)
goals2 = jnp.where(actual_g2_jax != -1, actual_g2_jax, sim_goals2)

points1 = jnp.where(goals1 > goals2, 3, jnp.where(goals1 == goals2, 1, 0))
points2 = jnp.where(goals2 > goals1, 3, jnp.where(goals1 == goals2, 1, 0))

gd1, gd2 = goals1 - goals2, goals2 - goals1

M1 = jnp.zeros((48, 72)).at[t1_idx, jnp.arange(72)].set(1)
M2 = jnp.zeros((48, 72)).at[t2_idx, jnp.arange(72)].set(1)

total_points = jnp.dot(points1, M1.T) + jnp.dot(points2, M2.T)
total_gd = jnp.dot(gd1, M1.T) + jnp.dot(gd2, M2.T)

scores = total_points * 1000 + total_gd
group_scores = scores.reshape(N_sims, 12, 4)

group_ranks = jnp.argsort(jnp.argsort(-group_scores, axis=2), axis=2)

first_place_mask = group_ranks == 0
second_place_mask = group_ranks == 1
third_place_mask = group_ranks == 2

third_scores = jnp.where(third_place_mask, group_scores, -jnp.inf)
third_scores_per_group = jnp.max(third_scores, axis=2)

third_ranks_among_groups = jnp.argsort(jnp.argsort(-third_scores_per_group, axis=1), axis=1)
best_8_third_mask = third_ranks_among_groups < 8
best_8_third_mask_expanded = jnp.expand_dims(best_8_third_mask, axis=2)

advances_as_third = jnp.logical_and(third_place_mask, best_8_third_mask_expanded)

prob_1st = first_place_mask.reshape(N_sims, 48).mean(axis=0)
prob_2nd = second_place_mask.reshape(N_sims, 48).mean(axis=0)
prob_best_3rd = advances_as_third.reshape(N_sims, 48).mean(axis=0)
prob_pass = prob_1st + prob_2nd + prob_best_3rd

# 6. Clean Matplotlib Visualization (No Flags)
group_letters = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']
team_groups = [group_letters[i // 4] for i in range(48)]

results_df = pd.DataFrame({
    'Group': team_groups,
    'Team': tournament_teams,
    '1st Place (%)': np.array(prob_1st) * 100,
    '2nd Place (%)': np.array(prob_2nd) * 100,
    'Best 3rd (%)': np.array(prob_best_3rd) * 100,
    'Total Pass (%)': np.array(prob_pass) * 100
})

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

plt.style.use('seaborn-v0_8-whitegrid')
colors = ['#2ecc71', '#f1c40f', '#3498db']

for idx, group in enumerate(group_letters):
    ax = axes[idx]
    
    group_df = results_df[results_df['Group'] == group].sort_values(by='Total Pass (%)', ascending=True)
    
    y_pos = np.arange(len(group_df))
    height = 0.6
    
    b1 = ax.barh(y_pos, group_df['1st Place (%)'], height, color=colors[0], label='1st Place')
    b2 = ax.barh(y_pos, group_df['2nd Place (%)'], height, left=group_df['1st Place (%)'], color=colors[1], label='2nd Place')
    b3 = ax.barh(y_pos, group_df['Best 3rd (%)'], height, left=group_df['1st Place (%)'] + group_df['2nd Place (%)'], color=colors[2], label='Best 3rd Wildcard')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(group_df['Team'], fontsize=12, fontweight='bold')
    ax.set_title(f'Group {group}', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlim(0, 115) 
    ax.set_xlabel('Advancement Probability (%)', fontsize=10)
    
    # Annotate breakdown percentages inside the bars safely
    for container in [b1, b2, b3]:
        for rect in container:
            width = rect.get_width()
            if width > 6.0:  # Only label if segment is wide enough
                ax.text(rect.get_x() + width/2, rect.get_y() + rect.get_height()/2, 
                        f'{width:.0f}%', ha='center', va='center', color='black', 
                        fontsize=10, fontweight='bold', alpha=0.85)

    # Render Total Pass % explicitly outside the bars
    for i, total in enumerate(group_df['Total Pass (%)']):
        ax.text(total + 2, i, f'{total:.1f}%', va='center', ha='left', color='black', fontweight='bold', fontsize=12)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 1.02), frameon=True)

plt.tight_layout(pad=3.0)

# Generate date string dynamic suffix for today (e.g. "17_06")
date_suffix = today.strftime('%d_%m')

# Export Data & Clean Image to Working Directory with dynamic date names
csv_path = f"2026_wc_detailed_probabilities_{date_suffix}.csv"
img_path = f"2026_wc_probabilities_chart_{date_suffix}.png"
results_df.to_csv(csv_path, index=False)
plt.savefig(img_path, bbox_inches='tight', dpi=300)

print(f"\nDirectory Set To: {os.getcwd()}")
print(f"Data exported to: {csv_path}")
print(f"Chart exported to: {img_path}")

plt.show()